# Week 12B — Monocular Depth and 3D Foundation Models

**MCTA 4364 Machine Vision** · Session 2 of Week 12 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W12_3d_vision/W12B_monocular_depth_3d_foundation.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W12_3d_vision/W12B_monocular_depth_3d_foundation.ipynb)

Session A needed two calibrated cameras. Humans can judge depth with one eye closed, from perspective, size, occlusion and shading — cues a deep
network can learn from millions of images. Today: **monocular depth estimation** with a modern foundation model (Depth Anything V2), how to evaluate
it honestly against stereo ground truth, why a single image only gives depth **up to scale**, and the new **3D foundation models** (DUSt3R, MASt3R,
VGGT) that reconstruct scenes and cameras from ordinary photos.

### Learning outcomes
By the end of this session you will be able to:
1. List the monocular depth cues and explain the **scale ambiguity** of a single image.
2. Run a pretrained monocular depth network and distinguish **relative (affine-invariant)** from **metric** depth.
3. **Align** a relative prediction to ground truth (scale and shift, least squares) and evaluate with **AbsRel**, **RMSE** and **δ < 1.25**.
4. Turn a depth map into a point cloud with an assumed field of view, and recognise its limits for measurement.
5. Explain what 3D foundation models (DUSt3R/MASt3R/VGGT) and neural scene representations (NeRF, Gaussian splatting) add.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1 | Monocular cues and scale ambiguity |
| 10–35 | 2–3 | Depth Anything V2; alignment and metrics against stereo ground truth (interactive) |
| 35–50 | 4 | Metric depth; monocular point clouds |
| 50–68 | 5–6 | 3D foundation models; NeRF and Gaussian splatting; choosing a 3D approach |
| 68–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

**Hardware:** laptop CPU is enough for the small model (~25 M parameters, ~1–2 s per image); the weights (~100 MB) download once from Hugging Face.

> **Licences.** Depth Anything V2 **Small** is Apache-2.0 (the Base/Large/Giant models are CC BY-NC 4.0); DUSt3R/MASt3R are CC BY-NC-SA 4.0 (research);
> VGGT has its own licence — always check before commercial use.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "transformers": "transformers"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch

import partsim
from cvhelpers import SMOKE, check, interact, load_sample, opencv_sample, pipeline_diagram, quiz, show
from dlhelpers import hw_profile

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
S = 0.5
aloe = cv2.resize(load_sample("aloeL.jpg"), None, fx=S, fy=S, interpolation=cv2.INTER_AREA)
GT_DISP = cv2.resize(load_sample("aloeGT.png", gray=True).astype(np.float32), None, fx=S, fy=S, interpolation=cv2.INTER_NEAREST) * S
F_PX, B_MM, DMIN = 3740.0 * S, 160.0, 270.0 * S                        # Middlebury aloe calibration (Week 12A)
VALID = GT_DISP > 0
GT_DEPTH = np.where(VALID, F_PX * B_MM / (GT_DISP + DMIN), np.nan)       # metric ground-truth depth (mm)
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi"))); _, street = cap.read(); cap.release()
images = {"aloe (Middlebury)": aloe, "street (vtest.avi)": street, "messi5.jpg": load_sample("messi5.jpg"), "conveyor (partsim)": partsim.make_scene(4)[0]}
print(f"aloe ground truth: {VALID.mean():.0%} of pixels, depth {np.nanmin(GT_DEPTH):.0f}-{np.nanmax(GT_DEPTH):.0f} mm")

## 1. Depth from one image: cues and ambiguity

**Monocular cues** used by people (and learned by networks): linear **perspective** (parallel lines converge), **relative size** and familiar size,
**occlusion**, **texture gradient**, **height in the image** (for objects on the ground), **shading**, atmospheric haze and **defocus**.

All of them are statistical: they work for typical scenes. Geometrically, one image cannot tell a model car at 1 m from a real car at 10 m — the
projection $x = fX/Z$ is unchanged if $X$ and $Z$ are scaled together. A monocular network therefore predicts either
- **relative depth** (affine-invariant: correct up to an unknown **scale and shift**, often as *inverse* depth / disparity), or
- **metric depth**, by learning typical object sizes and using the camera's focal length — an estimate, not a measurement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
X = np.array([[-1, 0], [1, 0], [1, 2], [-1, 2], [-1, 0]], float)               # a 2 m box, 10 m away
for s, col, lab in [(1.0, "#1a73e8", "real car: 2 m wide at 10 m"), (0.1, "#d93025", "toy car: 0.2 m wide at 1 m")]:
    Z = 10 * s
    axes[0].plot(X[:, 0] * s, np.full(len(X), Z) + X[:, 1] * s, color=col, lw=2.5, label=lab)
    axes[0].plot([0, X[0, 0] * s], [0, Z], color=col, ls=":"); axes[0].plot([0, X[1, 0] * s], [0, Z], color=col, ls=":")
axes[0].plot(0, 0, "k^", ms=12); axes[0].set_xlabel("X (m)"); axes[0].set_ylabel("depth Z (m)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title("top view: both objects fill exactly the same rays")
axes[1].bar(["real car", "toy car"], [1.0 * 2 / 10 * 1000, 0.2 / 1 * 1000], color=["#1a73e8", "#d93025"])
axes[1].set_ylabel("image width (px) for f = 1000 px"); axes[1].set_title("...so their images are identical: x = f X / Z")
plt.suptitle("Figure 1.1 - The scale ambiguity of a single image", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Depth Anything V2

**Depth Anything** (Yang et al., 2024) trains a ViT encoder (initialised from **DINOv2**) with a DPT decoder on a small set of precise *synthetic*
depth labels, then distils it into students on **62 million** unlabelled real images pseudo-labelled by the teacher — again the data engine of
Week 9A. It outputs **relative inverse depth** (large = near). We use the **Small** model via Hugging Face `transformers`.

In [ ]:
def load_depth_model():
    try:
        from transformers import pipeline
        return pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0 if HW["cuda"] else -1)
    except Exception as exc:                                   # no internet / Hugging Face unreachable
        print(f"WARNING: could not load Depth Anything V2 ({type(exc).__name__}). Falling back to a SIMULATED prediction "
              "(ground truth + distortions) so the rest of the notebook runs; results below are NOT from a network.")
        return None


depth_model = load_depth_model()


def predict_inverse_depth(img_bgr):
    """Relative inverse depth (larger = nearer), resized to the image."""
    if depth_model is not None:
        from PIL import Image
        out = depth_model(Image.fromarray(img_bgr[..., ::-1]))
        pred = out["predicted_depth"]
        pred = pred.squeeze().cpu().numpy() if torch.is_tensor(pred) else np.asarray(pred)
        return cv2.resize(pred.astype(np.float32), img_bgr.shape[1::-1], interpolation=cv2.INTER_CUBIC)
    # fallback: an affine distortion of a smoothed version of the true inverse depth (aloe) or of image brightness (others)
    base = (GT_DISP + DMIN) if img_bgr.shape[:2] == aloe.shape[:2] else cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    base = cv2.GaussianBlur(np.where(base > 0, base, np.median(base)), (0, 0), 3)
    return 0.37 * base + 12.0 + np.random.default_rng(0).normal(0, 1.0, base.shape).astype(np.float32)


preds = {}
t0 = time.perf_counter()
for name, im in images.items():
    preds[name] = predict_inverse_depth(im)
print(f"{len(preds)} images in {time.perf_counter() - t0:.1f} s")
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for k, (name, im) in enumerate(images.items()):
    axes[0, k].imshow(im[..., ::-1]); axes[0, k].set_title(name)
    axes[1, k].imshow(preds[name], cmap="Spectral_r"); axes[1, k].set_title("predicted relative inverse depth (red = near)")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 2.1 - Monocular depth from single images", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Evaluating relative depth honestly

A relative prediction $\hat{r}$ (inverse depth) must first be **aligned** to the ground truth $r^*$ with a scale $s$ and shift $t$ found by least
squares on the valid pixels (Ranftl et al., 2020):

$$(s, t) = \arg\min_{s,t} \sum_i \big(s\,\hat r_i + t - r^*_i\big)^2, \qquad \hat Z_i = 1 / (s\,\hat r_i + t).$$

Then the standard metrics on depth:

$$\text{AbsRel} = \frac{1}{N}\sum \frac{|\hat Z - Z^*|}{Z^*}, \qquad \text{RMSE} = \sqrt{\frac{1}{N}\sum (\hat Z - Z^*)^2}, \qquad
\delta_{1} = \frac{1}{N}\#\left\{\max\Big(\frac{\hat Z}{Z^*}, \frac{Z^*}{\hat Z}\Big) < 1.25\right\}.$$

We compare Depth Anything with the **stereo SGBM** result of Session A on the aloe scene.

In [ ]:
def align_scale_shift_ref(pred, target, mask):
    A = np.stack([pred[mask], np.ones(mask.sum())], 1)
    (s, t), *_ = np.linalg.lstsq(A, target[mask], rcond=None)
    return s, t


def depth_metrics_ref(pred_depth, gt_depth, mask):
    p, g = pred_depth[mask], gt_depth[mask]
    ratio = np.maximum(p / g, g / p)
    return dict(AbsRel=float(np.mean(np.abs(p - g) / g)), RMSE=float(np.sqrt(np.mean((p - g) ** 2))), delta1=float(np.mean(ratio < 1.25)))


gt_inv = np.where(VALID, 1.0 / GT_DEPTH, 0)
s_, t_ = align_scale_shift_ref(preds["aloe (Middlebury)"], gt_inv, VALID)
mono_depth = 1.0 / np.clip(s_ * preds["aloe (Middlebury)"] + t_, 1e-6, None)
sg = cv2.StereoSGBM_create(0, 128, 5, P1=8 * 3 * 25, P2=32 * 3 * 25, uniquenessRatio=10, speckleWindowSize=100, speckleRange=2,
                           mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
aloeR = cv2.resize(load_sample("aloeR.jpg"), None, fx=S, fy=S, interpolation=cv2.INTER_AREA)
d_sg = sg.compute(aloe, aloeR).astype(np.float32) / 16
stereo_depth = np.where(d_sg > 0, F_PX * B_MM / (d_sg + DMIN), np.nan)
m_both = VALID & np.isfinite(stereo_depth)
res_mono, res_stereo = depth_metrics_ref(mono_depth, GT_DEPTH, m_both), depth_metrics_ref(stereo_depth, GT_DEPTH, m_both)
fig, axes = plt.subplots(1, 4, figsize=(24, 4.8))
vmin, vmax = np.nanpercentile(GT_DEPTH, 2), np.nanpercentile(GT_DEPTH, 98)
for ax, (name, dep) in zip(axes[:3], [("ground truth (structured light)", GT_DEPTH), ("stereo SGBM (Session A)", stereo_depth),
                                      ("monocular, scale + shift aligned", np.where(VALID, mono_depth, np.nan))]):
    im_ = ax.imshow(dep, cmap="turbo", vmin=vmin, vmax=vmax); ax.set_title(name); ax.axis("off")
plt.colorbar(im_, ax=axes[2], fraction=0.04, label="depth (mm)")
names_ = ["AbsRel", "delta1"]
xs = np.arange(2)
axes[3].bar(xs - 0.2, [res_stereo[k] for k in names_], 0.4, label="stereo", color="#1a73e8")
axes[3].bar(xs + 0.2, [res_mono[k] for k in names_], 0.4, label="monocular (aligned)", color="#f29900")
axes[3].set_xticks(xs); axes[3].set_xticklabels(["AbsRel (lower is better)", "delta < 1.25 (higher is better)"]); axes[3].legend(); axes[3].grid(alpha=0.3, axis="y")
plt.suptitle("Figure 3.1 - Stereo vs monocular depth on the aloe scene (same valid pixels)", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"alignment: scale {s_:.3g}, shift {t_:.3g}")
for k in ["AbsRel", "RMSE", "delta1"]:
    print(f"{k:<7} stereo {res_stereo[k]:8.3f} | monocular (aligned) {res_mono[k]:8.3f}")

### 🎛️ Interactive: what if the scale is wrong?
Without alignment (e.g. a metric model with the wrong focal length), the whole depth map is scaled. Watch the metrics.

In [ ]:
def scale_demo(scale_error=1.0, shift_mm=0.0):
    dep = np.where(m_both, mono_depth * scale_error + shift_mm, np.nan)
    r = depth_metrics_ref(np.nan_to_num(dep, nan=1.0), GT_DEPTH, m_both)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    axes[0].imshow(dep, cmap="turbo", vmin=vmin, vmax=vmax); axes[0].axis("off")
    axes[0].set_title(f"scale x{scale_error:.2f}, shift {shift_mm:+.0f} mm: AbsRel {r['AbsRel']:.3f}, delta1 {r['delta1']:.2f}")
    axes[1].hist((dep - GT_DEPTH)[m_both], 80, color="#f29900"); axes[1].axvline(0, color="k")
    axes[1].set_xlabel("depth error (mm)"); axes[1].set_title("error distribution")
    plt.show()


interact(scale_demo, scale_error=widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.05), shift_mm=widgets.FloatSlider(value=0, min=-300, max=300, step=10))

## 4. Metric depth and monocular point clouds

**Metric** monocular models (Depth Pro, UniDepth, Metric3D v2, Depth Anything V2-Metric) output millimetres directly. They need, or estimate, the
camera's focal length — which is why a phone photo's EXIF data matters. Typical errors are 5–15% of the distance: fine for robot navigation,
scene understanding and AR; **not** a substitute for a calibrated 3D sensor when you must measure a part to 0.1 mm.

With a depth map and the intrinsics (or a guessed horizontal field of view, $f = \dfrac{W/2}{\tan(\mathrm{FOV}/2)}$) we can back-project a
point cloud, exactly as in Session A.

In [ ]:
def fov_to_focal(width, fov_deg):
    return width / 2 / np.tan(np.deg2rad(fov_deg) / 2)


name = "street (vtest.avi)"
inv = preds[name]
rel_depth = 1.0 / np.clip((inv - inv.min()) / (inv.max() - inv.min()) + 0.05, 1e-3, None)      # arbitrary scale: shape only
h_, w_ = rel_depth.shape
f_guess = fov_to_focal(w_, 60)
v_, u_ = np.mgrid[0:h_, 0:w_]
pts = np.stack([(u_ - w_ / 2) * rel_depth / f_guess, (v_ - h_ / 2) * rel_depth / f_guess, rel_depth], -1).reshape(-1, 3)
cols = street[..., ::-1].reshape(-1, 3)
sel = np.arange(len(pts))[:: (60 if SMOKE else 12)]
keep = pts[sel, 2] < np.percentile(pts[:, 2], 97)
fig = go.Figure(go.Scatter3d(x=pts[sel][keep, 0], y=pts[sel][keep, 2], z=-pts[sel][keep, 1], mode="markers",
                             marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in cols[sel][keep]])))
fig.update_layout(height=520, margin=dict(l=0, r=0, t=40, b=0), scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="depth", zaxis_title="-Y"),
                  title="Figure 4.1 - A point cloud from ONE image (relative scale, assumed 60 degree field of view): drag to rotate")
fig.show()

## 5. 3D foundation models

Classical 3D reconstruction from photos (**structure from motion**, e.g. COLMAP) chains Week 5 and Week 12A: features → matching → RANSAC → camera
poses → triangulation → bundle adjustment. It is accurate but brittle with few images, low texture or large viewpoint changes.

**3D foundation models** replace the chain with one network that regresses geometry directly:

| model | input | output | notes |
|---|---|---|---|
| **DUSt3R** (2024) | 2+ uncalibrated images | a **pointmap** per image (a 3D point for every pixel, in a common frame) + confidence | poses and intrinsics are recovered from the pointmaps |
| **MASt3R** (2024) | image pairs | pointmaps + dense local features for matching | metric scale; strong matcher for SfM |
| **VGGT** (2025) | 1 to hundreds of images | cameras, depth maps, pointmaps, point tracks in one forward pass | seconds instead of minutes of SfM |
| **Depth Anything V2 / Depth Pro** | one image | relative / metric depth | this session |

Scene **representations** for rendering new views:
- **NeRF** (2020): an MLP maps a 3D point and viewing direction to colour and density; images are rendered by volume integration. Slow but photorealistic.
- **3D Gaussian splatting** (2023): millions of 3D Gaussians rasterised in real time; now the standard for digital twins of rooms and factories.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(22, 4.8))
pipeline_diagram(["photos", "features\n(SIFT, W05)", "matching +\nRANSAC (W05B)", "camera poses\n(essential matrix)", "triangulation\n(W12A)",
                  "bundle\nadjustment", "sparse / dense\n3D"], colors=["#f1f3f4"] + ["#e8f0fe"] * 5 + ["#e6f4ea"], ax=axes[0], fontsize=9)
axes[0].set_title("classical structure from motion (COLMAP): many hand-designed stages")
pipeline_diagram(["photos", "one transformer\n(DUSt3R / MASt3R / VGGT)", "pointmaps +\nconfidence", "cameras, depth,\ntracks"],
                 colors=["#f1f3f4", "#fef7e0", "#e6f4ea", "#e6f4ea"], ax=axes[1], fontsize=10)
axes[1].set_title("3D foundation model: geometry regressed directly, in one or a few forward passes")
plt.suptitle("Figure 5.1 - From a pipeline to a single network", fontweight="bold")
plt.tight_layout(); plt.show()

### Optional: run a 3D foundation model
These models are large (0.5–5 GB) and fastest on a GPU with ≥ 8 GB. With a 4 GB laptop GPU, run them in Colab. Two practical routes:
- **VGGT**: `pip install vggt` from its GitHub repository, then `VGGT.from_pretrained("facebook/VGGT-1B")` and pass 2–20 photos of an object taken
  while walking around it; export the point cloud to `.ply`.
- **MASt3R-SfM / DUSt3R**: clone the repository and run its Gradio demo on your photos; compare the reconstruction with COLMAP.

Take 10 photos of a textured object on your desk and try one of them (see the challenge).

## 6. Choosing a 3D approach for a machine-vision task

| requirement | best choice |
|---|---|
| sub-millimetre heights, inline, known geometry | laser triangulation or structured light (Week 12A) |
| bin picking at 0.5–1.5 m, textureless parts | active stereo or structured light |
| outdoor robot, long range | stereo + LiDAR |
| rough depth for scene understanding, obstacle avoidance, AR, from existing cameras | **monocular depth** (metric model) |
| digital twin of a cell from a phone video | **VGGT / MASt3R + Gaussian splatting** |

Monocular networks are also combined with measurement sensors: they **fill holes** in stereo depth (textureless and occluded areas) and upsample
sparse LiDAR, using the sensor as metric anchor (exactly the scale-and-shift alignment of Section 3).

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Scale-and-shift alignment
Return `(s, t)` minimising $\sum_i (s\,p_i + t - q_i)^2$ over the pixels where `mask` is True (use `np.linalg.lstsq`).

In [ ]:
def align_scale_shift(pred, target, mask):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    r = np.random.default_rng(0)
    p = r.uniform(0, 1, (20, 30)); q = 3.0 * p + 0.5
    m = np.ones_like(p, bool); m[:5] = False; q[:5] = 99
    s, t = align_scale_shift(p, q, m)
    assert abs(s - 3.0) < 1e-9 and abs(t - 0.5) < 1e-9, f"expected (3, 0.5), got ({s:.4f}, {t:.4f})"
    s2, t2 = align_scale_shift(preds["aloe (Middlebury)"], gt_inv, VALID)
    assert abs(s2 - s_) < 1e-6 * max(1, abs(s_)) and abs(t2 - t_) < 1e-6 * max(1, abs(t_)), "differs from Section 3"


check("E1 align_scale_shift", _test_e1)

### E2. Depth metrics
Return a dict with `AbsRel`, `RMSE` and `delta1` (Section 3) over the pixels where `mask` is True.

In [ ]:
def depth_metrics(pred_depth, gt_depth, mask):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    gt = np.array([1.0, 2.0, 4.0, 10.0]); pr = np.array([1.1, 2.0, 2.0, 10.0]); m = np.array([True, True, True, False])
    r = depth_metrics(pr, gt, m)
    assert abs(r["AbsRel"] - (0.1 + 0 + 0.5) / 3) < 1e-9, f"AbsRel wrong: {r['AbsRel']}"
    assert abs(r["RMSE"] - np.sqrt((0.01 + 0 + 4) / 3)) < 1e-9, f"RMSE wrong: {r['RMSE']}"
    assert abs(r["delta1"] - 2 / 3) < 1e-9, f"delta1 wrong: {r['delta1']}"
    r2 = depth_metrics(mono_depth, GT_DEPTH, m_both)
    assert all(abs(r2[k] - res_mono[k]) < 1e-6 for k in r2), "differs from Section 3"


check("E2 depth_metrics", _test_e2)

### E3. Field of view to focal length
Return the focal length in pixels for an image `width` and a **horizontal** field of view `fov_deg` (Section 4), and the inverse function.

In [ ]:
def focal_from_fov(width, fov_deg):
    # TODO: your code here
    raise NotImplementedError


def fov_from_focal(width, f_px):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    assert abs(focal_from_fov(1000, 90) - 500) < 1e-9, "90 degrees over 1000 px -> f = 500 px"
    assert abs(fov_from_focal(640, focal_from_fov(640, 63.0)) - 63.0) < 1e-9, "the two functions must be inverses"
    assert abs(fov_from_focal(2 * F_PX, F_PX) - 90) < 1e-9, "f = W/2 means a 90 degree field of view"


check("E3 focal / fov", _test_e3)

### E4. Where does monocular depth fail? (no self-check)
Compute the per-pixel relative error map of the aligned monocular depth on the aloe scene. Where is it largest (thin leaves, depth edges, the
background)? Then evaluate the **conveyor scene**: can the network see the few-millimetre height differences of Week 12A? Explain with Section 1.

In [ ]:
# E4: your code here

## 8. Challenge: phone-photo 3D
Take 10–20 overlapping photos of a textured object (a shoe, a toolbox) while walking around it. Reconstruct it (a) with COLMAP or with OpenCV
(Week 5 matching + `cv2.recoverPose` + `cv2.triangulatePoints` for two views) and (b) with VGGT or MASt3R in Colab. Compare the time, the number of
points, and the scale. How would you give the reconstruction a metric scale? (Hint: an object of known size, or a chessboard from Week 2.)

In [ ]:
# Your challenge code here

## 9. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Why can a single image not give metric depth by geometry alone?',
        "options": [
            'Images are 8-bit',
            'Scaling the scene and its distance together gives the same image',
            'Lenses distort',
            'Colour is lost',
        ],
        "answer": 'd7c18d34d1',
        "explain": 'x = fX/Z is invariant to scaling X and Z.',
    },
    {
        "q": "What does 'affine-invariant (relative) depth' mean?",
        "options": [
            'Depth in millimetres',
            'Depth correct up to an unknown scale and shift',
            'Depth only at edges',
            'Depth for affine cameras',
        ],
        "answer": '247e9d2f43',
        "explain": 'It must be aligned before metric evaluation.',
    },
    {
        "q": 'How are scale and shift found when evaluating relative depth?',
        "options": [
            'Guessed',
            'Least squares against the ground truth on valid pixels',
            'From the image size',
            'From the colour histogram',
        ],
        "answer": '6576731f10',
        "explain": 'Then AbsRel, RMSE and delta1 are computed.',
    },
    {
        "q": 'A prediction has delta1 = 0.95. What does it mean?',
        "options": [
            '95% error',
            '95% of pixels are within a factor 1.25 of the true depth',
            'The RMSE is 0.95 m',
            'Scale 0.95',
        ],
        "answer": '61c989bcd3',
        "explain": 'max(pred/gt, gt/pred) < 1.25.',
    },
    {
        "q": 'What does DUSt3R predict for each input image?',
        "options": [
            'A class label',
            'A pointmap: a 3D point for every pixel in a common frame',
            'A mask',
            'A caption',
        ],
        "answer": '614b880f77',
        "explain": 'Camera poses follow from the pointmaps.',
    },
    {
        "q": 'Which is the best choice to measure part heights to 0.1 mm inline?',
        "options": [
            'Monocular depth network',
            'Laser triangulation / structured light',
            'Gaussian splatting',
            'A single RGB photo',
        ],
        "answer": 'b09a7fcc9c',
        "explain": 'Monocular depth is an estimate, not a measurement.',
    },
    {
        "q": 'What does 3D Gaussian splatting represent a scene with?',
        "options": [
            'A mesh only',
            'Millions of 3D Gaussians rasterised in real time',
            'A depth map',
            'An MLP only',
        ],
        "answer": '1d92f58a36',
        "explain": 'NeRF uses an MLP with volume rendering; splatting is much faster.',
    },
], title='Week 12B quiz')

## 10. Exit ticket (reflection)
1. Name one task in industry where monocular depth is good enough, and one where it is dangerous to rely on it.
2. How would you combine a stereo camera and a monocular network to get a complete, metric depth map?
3. What did Depth Anything learn from, and why does that matter for industrial images?

## 11. What's new (2025–26)

> - **Monocular depth**: Depth Anything V2 and its video version (consistent over time), Depth Pro (sharp metric depth in 0.3 s), MoGe and UniDepth V2
>   (metric depth + camera intrinsics from one image), and diffusion-based estimators (Marigold) for fine detail.
> - **Feed-forward 3D**: VGGT (CVPR 2025 best paper), MASt3R-SLAM, π³ and similar models make multi-view reconstruction a forward pass; they are being
>   integrated into robotics and SfM tools.
> - **Gaussian splatting** everywhere: feed-forward splatting from a few images, 4D (dynamic) splats, and digital twins of production lines.

## 12. References and further exploration
- Eigen, D., Puhrsch, C. & Fergus, R. (2014). *Depth map prediction from a single image using a multi-scale deep network.* NeurIPS. [arXiv:1406.2283](https://arxiv.org/abs/1406.2283)
- Ranftl, R. et al. (2020). *Towards robust monocular depth estimation: mixing datasets for zero-shot cross-dataset transfer* (MiDaS). TPAMI. [arXiv:1907.01341](https://arxiv.org/abs/1907.01341)
- Yang, L. et al. (2024). *Depth Anything V2.* NeurIPS. [arXiv:2406.09414](https://arxiv.org/abs/2406.09414)
- Bochkovskii, A. et al. (2025). *Depth Pro: Sharp monocular metric depth in less than a second.* ICLR. [arXiv:2410.02073](https://arxiv.org/abs/2410.02073)
- Oquab, M. et al. (2024). *DINOv2: Learning robust visual features without supervision.* TMLR. [arXiv:2304.07193](https://arxiv.org/abs/2304.07193)
- Schönberger, J. & Frahm, J.-M. (2016). *Structure-from-motion revisited* (COLMAP). CVPR. [doi:10.1109/CVPR.2016.445](https://doi.org/10.1109/CVPR.2016.445)
- Wang, S. et al. (2024). *DUSt3R: Geometric 3D vision made easy.* CVPR. [arXiv:2312.14132](https://arxiv.org/abs/2312.14132)
- Leroy, V., Cabon, Y. & Revaud, J. (2024). *Grounding image matching in 3D with MASt3R.* ECCV. [arXiv:2406.09756](https://arxiv.org/abs/2406.09756)
- Wang, J. et al. (2025). *VGGT: Visual geometry grounded transformer.* CVPR. [arXiv:2503.11651](https://arxiv.org/abs/2503.11651)
- Mildenhall, B. et al. (2020). *NeRF: Representing scenes as neural radiance fields for view synthesis.* ECCV. [arXiv:2003.08934](https://arxiv.org/abs/2003.08934)
- Kerbl, B. et al. (2023). *3D Gaussian splatting for real-time radiance field rendering.* ACM TOG 42(4). [arXiv:2308.04079](https://arxiv.org/abs/2308.04079)
- Hugging Face *depth estimation* task guide. [huggingface.co/docs/transformers/tasks/monocular_depth_estimation](https://huggingface.co/docs/transformers/tasks/monocular_depth_estimation)

**Data:** `aloeL.jpg`, `aloeR.jpg`, `aloeGT.png`, `vtest.avi`, `messi5.jpg` from the OpenCV sample data; conveyor scene from `partsim.py`.

## 13. Key takeaways
- A single image gives depth **up to scale**; networks learn cues and priors, so their depth is an **estimate**.
- Relative predictions must be **aligned** (scale + shift) before evaluation with **AbsRel, RMSE, δ₁**; metric models still need the focal length.
- Monocular depth complements, not replaces, measurement sensors; it fills holes and gives 3D where only one camera exists.
- **3D foundation models** (DUSt3R, MASt3R, VGGT) regress cameras and pointmaps directly; **NeRF / Gaussian splatting** render reconstructed scenes.

**Next week (13A):** vision–language models — describing, querying and **auto-labelling** images with text.